## This notebook adds the enformer prediction class (currently MAX over all cell-types) to the to be merged variant files
- First merging variant files (for each variant to which cell-type and variant-type (common, rare, ultra-rare, singleton) it belongs to)
- Merge all the enformer files (only the raw files exist) for each cell-type and variant file to the merged variant files and compute the enformer class   
- path: /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/<gene_list>/enformer/<gene_list>.<variant_type>_680_columns.tsv.gz
- gene_list = ['cardiac', 'cava', 'neuro', 'random']
- variant_type = ['ultra_rare', 'singleton']
- read file into pandas, get following columns
  - #CHROM	POS	ID	REF	ALT	DNase_max	INDEX_max	QUAL	FILTER	INFO	gene_set	variant_type	enformer_class

In [7]:
import pandas as pd
import os
import yaml 

config_path = "../../global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

### Sanity check if all files exist

In [2]:
gene_list = ['cardiac', 'cava', 'neuro', 'random']
variant_type_list = ['ultra-rare', 'singleton']
for gene_type in gene_list:
    for variant_type in variant_type_list:
        enformer_file_name = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/enformer/{gene_type}.{variant_type}_680_columns.tsv.gz'
        prioritized_variant_file = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/variants/{gene_type}.{variant_type}_variants.vcf'
        print(enformer_file_name)
        print(os.path.exists(enformer_file_name))
        print(prioritized_variant_file)
        print(os.path.exists(prioritized_variant_file))

/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/enformer/cardiac.ultra-rare_680_columns.tsv.gz
True
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/variants/cardiac.ultra-rare_variants.vcf
True
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/enformer/cardiac.singleton_680_columns.tsv.gz
True
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/variants/cardiac.singleton_variants.vcf
True
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cava/enformer/cava.ultra-rare_680_columns.tsv.gz
True
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cava/variants/cava.ultra-rare_variants.vcf
True
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cava/enformer/cava.singleton_680_columns.t

In [ ]:
## Helpful code snippets

# for one example
# load all files in the folder '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/' with the extention '.tsv.gz'
# /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/<gene_list>/enformer/<gene_list>.<variant_type>_680_columns.tsv.gz
# /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/<gene_list>/variants/<gene_list>.<variant_type>_variants.vcf

gene_list = ['cardiac', 'cava', 'neuro', 'random']
# gene_list = ['neuro']
variant_type_list = ['common','rare','ultra-rare', 'singleton']
# variant_type_list = ['ultra-rare']

for gene_type in gene_list:
    for variant_type in variant_type_list:
        print(f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants/{gene_type}.{variant_type}_variants.tsv')
        # enformer_file_name = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/enformer/test_{gene_type}.{variant_type}_680_columns.tsv.gz'
        # prioritized_variant_file = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/variants/{gene_type}.{variant_type}_variants.vcf'
        # print(enformer_file_name)
        # print(prioritized_variant_file)
        # enformer_file = pd.read_csv(enformer_file_name, sep='\t')
        # variant_file = pd.read_csv(prioritized_variant_file, sep='\t', comment='#', header=None)
        # variant_file.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']
        
        # break

### Start combining enformer and variant vcf (scripts should be used because notebook might crash due to memory issues)
- preprocess enformer files 
  - computing max over all cell-types
  - remove unused columns
  - save to new file
- preprocess variant vcfs 
  - add cell-type and variant-type columns
  - merge variant files to one file
  - save to new file

In [ ]:
## write function which prepares the enformer files (getting max and dropping columns => storing again)

def get_max_and_column_enformer_file(enformer_file_path):
    """Returns the modified enformer file with max and max column"""
    enformer_file = pd.read_csv(enformer_file_path, sep='\t')
    # compute max and id of max as mohan did in combine_tsv_vcf.py
    enformer_df = enformer_file.iloc[:,:679]
    print(enformer_df.iloc[:,5:].head())
    enformer_df['DNase_max'] = enformer_df.iloc[:,5:].values.max(axis = 1)
    enformer_df['max_col'] = enformer_df.iloc[:,5:].idxmax(axis = 1)
    # drop all columns with 'DNASE' in name
    enformer_df = enformer_df.drop(enformer_df.filter(regex='DNASE').columns, axis=1)
    return enformer_df

def write_modified_enformer_file(gene_type, variant_type, enformer_out_name):
    enformer_file_name = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/enformer/{gene_type}.{variant_type}_680_columns.tsv.gz'
    enformer_df = get_max_and_column_enformer_file(enformer_file_name)
    print('Write to file: ', enformer_out_name)
    enformer_df.to_csv(enformer_out_name, sep='\t', index=False)

gene_list = ['cardiac', 'cava', 'neuro', 'random']
# gene_list = ['neuro']
variant_type_list = ['ultra-rare', 'singleton']

for gene_type in gene_list:
    for variant_type in variant_type_list:
        enformer_out_name = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/enformer/{gene_type}.{variant_type}_max_values.tsv'
        write_modified_enformer_file(gene_type, variant_type, enformer_out_name)

In [ ]:
# # one example: 
# gene_type = 'neuro'
# variant_type = 'ultra-rare'
# enformer_file_name = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/enformer/{gene_type}.{variant_type}_680_columns.tsv.gz'
# prioritized_variant_file = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/variants/{gene_type}.{variant_type}_variants.vcf'
# print(enformer_file_name)
# print(prioritized_variant_file)
# enformer_file = pd.read_csv(enformer_file_name, sep='\t')
# enformer_file.head()

# enformer_df.head()

/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/neuro/enformer/neuro.ultra-rare_680_columns.tsv.gz
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/neuro/variants/neuro.ultra-rare_variants.vcf


: 

: 

: 

In [8]:
# add to each vcf file additional columns: gene_type and variant_type and merge them together

gene_list = ['cardiac', 'cava', 'neuro', 'random']
variant_type_list = ['common', 'rare', 'ultra-rare', 'singleton']

outpath_list = []
for gene_type in gene_list:
    for variant_type in variant_type_list:
        prioritized_variant_file = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/variants/{gene_type}.{variant_type}_variants.vcf'
        print(prioritized_variant_file)
        variant_file = pd.read_csv(prioritized_variant_file, sep='\t', comment='#', header=None)
        variant_file.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']
        variant_file.head()
        # add additional columns
        variant_file['gene_type'] = gene_type
        variant_file['variant_type'] = variant_type
        variant_file.head()
        # write to different directory
        outdir = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants'
        output_name = f'{gene_type}.{variant_type}_variants.tsv'
        output_path = os.path.join(outdir, output_name)
        outpath_list.append(output_path)
        print('Write to file: ', output_path)
        # variant_file.to_csv(output_path, sep='\t', index=False)
        
# merge the list of files together
merged_df = pd.concat([pd.read_csv(file, sep='\t') for file in outpath_list])
merged_df.to_csv(config['files']['creating']['merged_variant_table'], sep='\t', index=False)

/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/variants/cardiac.common_variants.vcf
Write to file:  /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants/cardiac.common_variants.tsv
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/variants/cardiac.rare_variants.vcf
Write to file:  /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants/cardiac.rare_variants.tsv
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/variants/cardiac.ultra-rare_variants.vcf
Write to file:  /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants/cardiac.ultra-rare_variants.tsv
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/variants/cardiac.singleton_variants.vcf


In [20]:
# merged variant file 
merged_variants_file = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants/merged_prioritized_variants.tsv'
merged_variants = pd.read_csv(merged_variants_file, sep='\t')
merged_variants


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,gene_type,variant_type
0,chr1,3030162,PRDM16|ENSG00000142611.17|EH38E1312284|1-30301...,T,A,1,PASS,AF=0.247769;AC=37649,cardiac,common
1,chr1,3030855,PRDM16|ENSG00000142611.17|EH38E2779521|1-30308...,G,A,1,PASS,AF=0.0757297;AC=11520,cardiac,common
2,chr1,3033425,PRDM16|ENSG00000142611.17|EH38E1312290|1-30334...,A,C,1,AS_VQSR,AF=0.0564555;AC=4921,cardiac,common
3,chr1,3033673,PRDM16|ENSG00000142611.17|EH38E2779527|1-30336...,G,A,1,PASS,AF=0.170986;AC=25994,cardiac,common
4,chr1,3033770,PRDM16|ENSG00000142611.17|EH38E2779527|1-30337...,G,A,1,PASS,AF=0.171221;AC=26042,cardiac,common
...,...,...,...,...,...,...,...,...,...,...
71405,chr6,26284345,H1-3|ENSG00000124575.7|EH38E3697740|6-26284345...,G,A,1,PASS,AF=6.57903e-06;AC=1,random,singleton
71406,chr2,210478202,CPS1|ENSG00000021826.18|EH38E3397495|2-2104782...,C,T,1,PASS,AF=6.5697e-06;AC=1,random,singleton
71407,chr2,210477975,CPS1|ENSG00000021826.18|EH38E3397495|2-2104779...,G,C,1,PASS,AF=6.57315e-06;AC=1,random,singleton
71408,chr2,210478142,CPS1|ENSG00000021826.18|EH38E3397495|2-2104781...,T,A,1,PASS,AF=6.56858e-06;AC=1,random,singleton


In [ ]:
# merge enformer and write the not is na rows to another directory
gene_list = ['cardiac', 'cava', 'neuro', 'random']
variant_type_list = ['common','rare','ultra-rare', 'singleton']

for gene_type in gene_list:
    for variant_type in variant_type_list:
        # read variant file
        prioritized_variant_file = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants/{gene_type}.{variant_type}_variants.tsv'
        print(prioritized_variant_file)
        # read enformer max file
        # variant_file = pd.read_csv(prioritized_variant_file, sep='\t', comment='#', header=None)
        # variant_file.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']
        # variant_file.head()
        ## add additional columns
        variant_file['gene_type'] = gene_type
        variant_file['variant_type'] = variant_type
        variant_file.head()
        # write to different directory
        outdir = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants'
        output_name = f'{gene_type}.{variant_type}_variants.tsv'
        output_path = os.path.join(outdir, output_name)
        print('Write to file: ', output_path)
        variant_file.to_csv(output_path, sep='\t', index=False)

In [31]:
# groupby gene_type and variant_type and count (sanity check passed: 5000, 5000, 5000, 2500 => 17500 for singleton and ultra-rare)
merged_variants.groupby(['variant_type', 'gene_type'])['ID'].count()

variant_type  gene_type
common        cardiac       6460
              cava          1422
              neuro        11591
              random         509
rare          cardiac       5283
              cava          1136
              neuro         9562
              random         447
singleton     cardiac       5000
              cava          5000
              neuro         5000
              random        2500
ultra-rare    cardiac       5000
              cava          5000
              neuro         5000
              random        2500
Name: ID, dtype: int64

In [11]:
# one example
gene_type = 'cardiac'
variant_type = 'ultra-rare'
prioritized_variant_file = f'/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/{gene_type}/variants/{gene_type}.{variant_type}_variants.vcf'
print(prioritized_variant_file)
variant_file = pd.read_csv(prioritized_variant_file, sep='\t', comment='#', header=None)
variant_file.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']
variant_file.head()
# add additional columns
variant_file['gene_type'] = gene_type
variant_file['variant_type'] = variant_type
variant_file.head()
# write to different directory
outdir = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/merging_variants'
output_name = f'{gene_type}.{variant_type}_variants.tsv'
output_path = os.path.join(outdir, output_name)
print('Write to file: ', output_path)
variant_file.to_csv(output_path, sep='\t', index=False)

/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/cardiac/variants/cardiac.ultra-rare_variants.vcf


In [116]:


# read variant file and match with id all files have 8 lines of comments
variant_file = pd.read_csv(prioritized_variant_file, sep='\t', skiprows=8)
variant_file.head()
print(variant_file.shape[0])
# merge the two files: take variant_file and add DNase_max and max_col from enformer_df (with ID and id)
# merged_enformer_file = pd.merge(variant_file, enformer_df, left_on='ID', right_on='id', how='left')
# merged_enformer_file.head()
# print(merged_enformer_file[merged_enformer_file['Dnase_max'].isna()].shape[0])


/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/neuro/enformer/test_neuro.ultra-rare_680_columns.tsv.gz
/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/neuro/variants/neuro.ultra-rare_variants.vcf
5000


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chrom,pos,id,ref,alt,DNase_max,max_col
0,chr7,70660644,AUTS2|ENSG00000158321.19|EH38E3778477|7-706606...,C,T,1,PASS,AF=0.000348226;AC=53,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,chr2,63856432,UGP2|ENSG00000169764.16|EH38E3347620|2-6385643...,A,T,1,PASS,AF=1.31475e-05;AC=2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chr16,31064606,BCKDK|ENSG00000103507.14|EH38E3177440|16-31064...,C,G,1,PASS,AF=1.31384e-05;AC=2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,chr18,44729852,SETBP1|ENSG00000152217.20|EH38E3265867|18-4472...,G,A,1,PASS,AF=0.000164225;AC=25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,chr21,46425079,DIP2A|ENSG00000160305.18|EH38E3465123|21-46425...,G,T,1,PASS,AF=0.000729716;AC=111,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [118]:
# look for examples in enformer_file:
enformer_df.head()
enformer_df[enformer_df['id'] == 'AUTS2|ENSG00000158321.19|EH38E3778477|7-70660644-C-T']

,chrom,pos,id,ref,alt,DNase_max,max_col


In [89]:
# read merged file without singleton
merged_enformer_file = pd.read_csv('/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/projects/80K_MPRA/enformer_results/enformer_class/merged_enformer_class.tsv', sep='\t')
merged_enformer_file.head()

,#CHROM,POS,ID,REF,ALT,DNase_max,INDEX_max,QUAL,FILTER,INFO,gene_set,variant_type,enformer_class
0,chr20,37315989,SRC|ENSG00000197122.13|EH38E2110582|20-3731598...,G,A,3986.3271,392_DNASE:CD14-positive,1,PASS,AF=6.56987e-06;AC=1,cava,singleton,enformer_high
1,chr11,61381959,SDHAF2|ENSG00000167985.7|EH38E2960484|11-61381...,G,A,2724.2556,392_DNASE:CD14-positive,1,PASS,AF=6.57419e-06;AC=1,cava,singleton,enformer_high
2,chr14,65081632,MAX|ENSG00000125952.20|EH38E1721673|14-6508163...,G,C,2250.6520,392_DNASE:CD14-positive,1,PASS,AF=6.56935e-06;AC=1,cava,singleton,enformer_high
3,chr1,161350864,SDHC|ENSG00000143252.16|EH38E2843757|1-1613508...,T,C,2191.6277,392_DNASE:CD14-positive,1,PASS,AF=6.57194e-06;AC=1,cava,singleton,enformer_high
4,chr16,1991191,NTHL1|ENSG00000065057.9|EH38E3162644|16-199119...,G,A,2112.9138,"177_DNASE:CD8-positive,",1,PASS,AF=6.56814e-06;AC=1,cava,singleton,enformer_high
